# 왜 사용자가 준 문장이 저장소 프롬프트보다 결함을 더 많이 찾았나

**디코더 마스킹으로 읽는 검증 하네스의 맹점**

대상: `unity_local_mcp` v1.11.15 · 분석일: 2026-07-30

---

## 한 문단 요약

저장소가 스스로 쓴 프롬프트 9개는 하네스 결함 5개를 찾았고, 사용자가 준 문장 3개는
15개를 찾았다. 문장당 **0.56 대 5.0**이다. 원인은 표본 수가 아니라 **어휘**다.
추출기를 만든 사람이 프롬프트도 썼기 때문에, 저장소 프롬프트는 추출기가 이미 아는
표기만 쓴다. 이 구조는 트랜스포머 디코더의 **마스킹**과 같다 — 마스크된 위치는
오류를 내지 않고 조용히 제외되며, softmax는 남은 것들로 재정규화해 **확신에 찬 답**을
낸다. 하네스도 똑같이 했다. `ad키`를 못 본 채 방향키로 측정하고 `verified`를 찍었다.


---

## 1. 무슨 일이 있었나 — 숫자

<svg xmlns="http://www.w3.org/2000/svg" width="700" height="134" font-family="system-ui,Segoe UI,sans-serif"><text x="0" y="22" font-size="15" font-weight="600" fill="#1f2733">발견한 하네스 결함</text><text x="190" y="56" font-size="13" text-anchor="end" fill="#1f2733">저장소 프롬프트 9개</text><rect x="200" y="43" width="140" height="18" rx="3" fill="#8892a6"/><text x="348" y="57" font-size="12.5" fill="#555">5개</text><text x="190" y="90" font-size="13" text-anchor="end" fill="#1f2733">사용자 문장 3개</text><rect x="200" y="77" width="420" height="18" rx="3" fill="#2f6fdb"/><text x="628" y="91" font-size="12.5" fill="#555">15개</text><text x="0" y="126" font-size="11.5" fill="#666">문장 수는 1/3인데 결함은 3배다.</text></svg>

<svg xmlns="http://www.w3.org/2000/svg" width="700" height="134" font-family="system-ui,Segoe UI,sans-serif"><text x="0" y="22" font-size="15" font-weight="600" fill="#1f2733">문장 1개당 결함 (효율)</text><text x="190" y="56" font-size="13" text-anchor="end" fill="#1f2733">저장소 프롬프트</text><rect x="200" y="43" width="47" height="18" rx="3" fill="#8892a6"/><text x="255" y="57" font-size="12.5" fill="#555">0.56개</text><text x="190" y="90" font-size="13" text-anchor="end" fill="#1f2733">사용자 문장</text><rect x="200" y="77" width="420" height="18" rx="3" fill="#2f6fdb"/><text x="628" y="91" font-size="12.5" fill="#555">5.0개</text><text x="0" y="126" font-size="11.5" fill="#666">9배 차이.</text></svg>

| | 저장소 프롬프트 | 사용자 문장 |
|---|---:|---:|
| 고유 문장 | 9개 | 3개 |
| 발견한 결함 | 5개 | **15개** |
| 문장당 결함 | 0.56 | **5.0** |

### 1.1 코퍼스는 생각보다 작다

`prompts/`에는 파일이 43개 있지만 씬 경로만 바꾼 복제본이 대부분이다.

<svg xmlns="http://www.w3.org/2000/svg" width="700" height="202" font-family="system-ui,Segoe UI,sans-serif"><text x="0" y="22" font-size="15" font-weight="600" fill="#1f2733">코퍼스 압축</text><text x="190" y="56" font-size="13" text-anchor="end" fill="#1f2733">prompts/ 파일</text><rect x="200" y="43" width="420" height="18" rx="3" fill="#8892a6"/><text x="628" y="57" font-size="12.5" fill="#555">43개</text><text x="190" y="90" font-size="13" text-anchor="end" fill="#1f2733">씬 경로 제거 후 고유</text><rect x="200" y="77" width="117" height="18" rx="3" fill="#8892a6"/><text x="325" y="91" font-size="12.5" fill="#555">12개</text><text x="190" y="124" font-size="13" text-anchor="end" fill="#1f2733">└ 저장소 작성</text><rect x="200" y="111" width="87" height="18" rx="3" fill="#8892a6"/><text x="295" y="125" font-size="12.5" fill="#555">9개</text><text x="190" y="158" font-size="13" text-anchor="end" fill="#1f2733">└ 사용자 제공</text><rect x="200" y="145" width="29" height="18" rx="3" fill="#2f6fdb"/><text x="237" y="159" font-size="12.5" fill="#555">3개</text><text x="0" y="194" font-size="11.5" fill="#666">43개를 넣어도 실제로 검사한 문장은 12개다.</text></svg>

---

## 2. 결정적 차이는 어휘 다양성이다

추출기(`verification.VerificationSpec`)는 정규식으로 문자열을 본다. 그러므로 **같은
뜻을 다르게 적을 때마다 완전히 다른 코드 경로**다.

<svg xmlns="http://www.w3.org/2000/svg" width="640" height="522" font-family="system-ui,Segoe UI,sans-serif"><text x="0" y="20" font-size="15" font-weight="600" fill="#1f2733">같은 개념의 표기 × 누가 썼나</text><text x="298.0" y="52" font-size="12.5" text-anchor="middle" font-weight="600">저장소</text><text x="394.0" y="52" font-size="12.5" text-anchor="middle" font-weight="600">사용자</text><text x="0" y="82" font-size="12" fill="#667" font-weight="600">A/D 이동</text><text x="238" y="82" font-size="12.5" text-anchor="end" fill="#1f2733">A/D</text><rect x="256" y="67" width="84" height="19" rx="3" fill="#8892a6"/><text x="298.0" y="82" font-size="11.5" text-anchor="middle" fill="#fff">사용</text><rect x="352" y="67" width="84" height="19" rx="3" fill="#2f6fdb"/><text x="394.0" y="82" font-size="11.5" text-anchor="middle" fill="#fff">사용</text><text x="238" y="109" font-size="12.5" text-anchor="end" fill="#1f2733">ad키</text><rect x="256" y="94" width="84" height="19" rx="3" fill="#eef1f5"/><rect x="352" y="94" width="84" height="19" rx="3" fill="#2f6fdb"/><text x="394.0" y="109" font-size="11.5" text-anchor="middle" fill="#fff">사용</text><text x="238" y="136" font-size="12.5" text-anchor="end" fill="#1f2733">ad로</text><rect x="256" y="121" width="84" height="19" rx="3" fill="#eef1f5"/><rect x="352" y="121" width="84" height="19" rx="3" fill="#2f6fdb"/><text x="394.0" y="136" font-size="11.5" text-anchor="middle" fill="#fff">사용</text><text x="238" y="163" font-size="12.5" text-anchor="end" fill="#1f2733">A, D키</text><rect x="256" y="148" width="84" height="19" rx="3" fill="#eef1f5"/><rect x="352" y="148" width="84" height="19" rx="3" fill="#2f6fdb"/><text x="394.0" y="163" font-size="11.5" text-anchor="middle" fill="#fff">사용</text><text x="0" y="190" font-size="12" fill="#667" font-weight="600">점프 키</text><text x="238" y="190" font-size="12.5" text-anchor="end" fill="#1f2733">Space 점프</text><rect x="256" y="175" width="84" height="19" rx="3" fill="#8892a6"/><text x="298.0" y="190" font-size="11.5" text-anchor="middle" fill="#fff">사용</text><rect x="352" y="175" width="84" height="19" rx="3" fill="#eef1f5"/><text x="238" y="217" font-size="12.5" text-anchor="end" fill="#1f2733">space가</text><rect x="256" y="202" width="84" height="19" rx="3" fill="#eef1f5"/><rect x="352" y="202" width="84" height="19" rx="3" fill="#2f6fdb"/><text x="394.0" y="217" font-size="11.5" text-anchor="middle" fill="#fff">사용</text><text x="238" y="244" font-size="12.5" text-anchor="end" fill="#1f2733">space 키</text><rect x="256" y="229" width="84" height="19" rx="3" fill="#eef1f5"/><rect x="352" y="229" width="84" height="19" rx="3" fill="#2f6fdb"/><text x="394.0" y="244" font-size="11.5" text-anchor="middle" fill="#fff">사용</text><text x="0" y="271" font-size="12" fill="#667" font-weight="600">부스트</text><text x="238" y="271" font-size="12.5" text-anchor="end" fill="#1f2733">LeftShift</text><rect x="256" y="256" width="84" height="19" rx="3" fill="#8892a6"/><text x="298.0" y="271" font-size="11.5" text-anchor="middle" fill="#fff">사용</text><rect x="352" y="256" width="84" height="19" rx="3" fill="#eef1f5"/><text x="238" y="298" font-size="12.5" text-anchor="end" fill="#1f2733">좌쉬프트</text><rect x="256" y="283" width="84" height="19" rx="3" fill="#eef1f5"/><rect x="352" y="283" width="84" height="19" rx="3" fill="#2f6fdb"/><text x="394.0" y="298" font-size="11.5" text-anchor="middle" fill="#fff">사용</text><text x="238" y="325" font-size="12.5" text-anchor="end" fill="#1f2733">쉬프트 키</text><rect x="256" y="310" width="84" height="19" rx="3" fill="#eef1f5"/><rect x="352" y="310" width="84" height="19" rx="3" fill="#2f6fdb"/><text x="394.0" y="325" font-size="11.5" text-anchor="middle" fill="#fff">사용</text><text x="238" y="352" font-size="12.5" text-anchor="end" fill="#1f2733">Shift 키</text><rect x="256" y="337" width="84" height="19" rx="3" fill="#eef1f5"/><rect x="352" y="337" width="84" height="19" rx="3" fill="#2f6fdb"/><text x="394.0" y="352" font-size="11.5" text-anchor="middle" fill="#fff">사용</text><text x="0" y="379" font-size="12" fill="#667" font-weight="600">카메라</text><text x="238" y="379" font-size="12.5" text-anchor="end" fill="#1f2733">따라오</text><rect x="256" y="364" width="84" height="19" rx="3" fill="#8892a6"/><text x="298.0" y="379" font-size="11.5" text-anchor="middle" fill="#fff">사용</text><rect x="352" y="364" width="84" height="19" rx="3" fill="#eef1f5"/><text x="238" y="406" font-size="12.5" text-anchor="end" fill="#1f2733">추적</text><rect x="256" y="391" width="84" height="19" rx="3" fill="#eef1f5"/><rect x="352" y="391" width="84" height="19" rx="3" fill="#2f6fdb"/><text x="394.0" y="406" font-size="11.5" text-anchor="middle" fill="#fff">사용</text><text x="0" y="433" font-size="12" fill="#667" font-weight="600">새 씬</text><text x="238" y="433" font-size="12.5" text-anchor="end" fill="#1f2733">새 빈 씬</text><rect x="256" y="418" width="84" height="19" rx="3" fill="#8892a6"/><text x="298.0" y="433" font-size="11.5" text-anchor="middle" fill="#fff">사용</text><rect x="352" y="418" width="84" height="19" rx="3" fill="#eef1f5"/><text x="238" y="460" font-size="12.5" text-anchor="end" fill="#1f2733">새 씬</text><rect x="256" y="445" width="84" height="19" rx="3" fill="#8892a6"/><text x="298.0" y="460" font-size="11.5" text-anchor="middle" fill="#fff">사용</text><rect x="352" y="445" width="84" height="19" rx="3" fill="#2f6fdb"/><text x="394.0" y="460" font-size="11.5" text-anchor="middle" fill="#fff">사용</text><text x="238" y="487" font-size="12.5" text-anchor="end" fill="#1f2733">새씬</text><rect x="256" y="472" width="84" height="19" rx="3" fill="#eef1f5"/><rect x="352" y="472" width="84" height="19" rx="3" fill="#2f6fdb"/><text x="394.0" y="487" font-size="11.5" text-anchor="middle" fill="#fff">사용</text><text x="0" y="516" font-size="11.5" fill="#666">저장소는 개념마다 한 가지 표기만 쓴다 — 그 표기를 정한 사람이 프롬프트도 썼다.</text></svg>

- 저장소 9개 문장이 쓴 표기: **6종** — `A/D`, `LeftShift`, `Space 점프`, `따라오`, `새 빈 씬`, `새 씬`
- 사용자 3개 문장이 쓴 표기: **12종**
- 사용자만 밟은 표기: **10종** — `ad키`, `ad로`, `A, D키`, `space가`, `space 키`,
  `좌쉬프트`, `쉬프트 키`, `Shift 키`, `추적`, `새씬`

이 10종이 결함 목록과 거의 일대일로 대응한다.

---

## 3. 결함 20건 전체

각 항목의 실측 근거는 `docs/` 아래 해당 버전 문서에 있다.

| # | 출처 | 종류 | 결함 | 문서 |
|---:|---|---|---|---|
| 1 | 저장소 | 판정 | 격자 요청이 검사 0개로 `verified` — 공집합 성공 재발 | v1.11.13 |
| 2 | 저장소 | 구조 | 호스트 파일 도구가 프로젝트 정체성 가드를 우회 | v1.11.13 |
| 3 | 저장소 | 판정 | 빈 씬 템플릿에 카메라가 없어 전 검사가 측정 전 차단 | v1.11.12 |
| 4 | 저장소 | 게이트 | 미정의 Ground 태그 검사가 점프 요청 안에만 걸려 있음 | v1.11.12 |
| 5 | 저장소 | 판정 | Play 종료 후 런타임 오류를 컴파일 오류로 계산 → rollback | v1.11.12 |
| 6 | **사용자** | 어휘 | `ad키` 미인식 → 하네스가 **방향키로** 측정 | v1.11.14 |
| 7 | **사용자** | 어휘 | 좌우 양방향 검사 누락 | v1.11.14 |
| 8 | **사용자** | 어휘 | `좌쉬프트` 미인식 → 부스트 검사 없음 | v1.11.14 |
| 9 | **사용자** | 어휘 | `추적` 미인식 → 카메라 검사 없음 | v1.11.14 |
| 10 | **사용자** | 안내 | 부스트 실패에 수정 안내가 아예 없음 | v1.11.14 |
| 11 | **사용자** | 안내 | 점프 실패가 배치(천장) 때문일 때 안내 없음 | v1.11.14 |
| 12 | **사용자** | 어휘 | `ad로` 미인식 (조사만 다름) | v1.11.14 |
| 13 | **사용자** | 구조 | 게이트와 측정이 어휘를 따로 보유 → 강제키 ≠ 측정키 | v1.11.14 |
| 14 | **사용자** | 어휘 | `새씬`(붙여쓰기) 미인식 → 새 씬 정책 전체가 꺼짐 | v1.11.14 |
| 15 | **사용자** | 어휘 | 게이트의 카메라 판정이 `추적`을 모름 | v1.11.14 |
| 16 | **사용자** | 구조 | `\b` 경계가 한글에서 깨짐 (`D로`의 d와 로 사이에 경계 없음) | v1.11.14 |
| 17 | **사용자** | 판정 | 플레이어에 붙은 **시점 카메라가 추종 검사를 통과** | v1.11.15 |
| 18 | **사용자** | 게이트 | `CameraController.cs`를 입력 스크립트로 오인해 **13회 차단** | v1.11.15 |
| 19 | **사용자** | 판정 | 부스트 상한 부재 — 0.5초에 140유닛 간 대시가 통과 | v1.11.15 |
| 20 | **사용자** | 안내 | 장애물이 측정을 막을 때 안내 없음 | v1.11.15 |

<svg xmlns="http://www.w3.org/2000/svg" width="700" height="406" font-family="system-ui,Segoe UI,sans-serif"><text x="0" y="22" font-size="15" font-weight="600" fill="#1f2733">결함 종류별 출처</text><text x="190" y="56" font-size="13" text-anchor="end" fill="#1f2733">어휘 · 저장소</text><rect x="200" y="43" width="2" height="18" rx="3" fill="#8892a6"/><text x="210" y="57" font-size="12.5" fill="#555">0건</text><text x="190" y="90" font-size="13" text-anchor="end" fill="#1f2733">어휘 · 사용자</text><rect x="200" y="77" width="420" height="18" rx="3" fill="#2f6fdb"/><text x="628" y="91" font-size="12.5" fill="#555">7건</text><text x="190" y="124" font-size="13" text-anchor="end" fill="#1f2733">안내 · 저장소</text><rect x="200" y="111" width="2" height="18" rx="3" fill="#8892a6"/><text x="210" y="125" font-size="12.5" fill="#555">0건</text><text x="190" y="158" font-size="13" text-anchor="end" fill="#1f2733">안내 · 사용자</text><rect x="200" y="145" width="180" height="18" rx="3" fill="#2f6fdb"/><text x="388" y="159" font-size="12.5" fill="#555">3건</text><text x="190" y="192" font-size="13" text-anchor="end" fill="#1f2733">구조 · 저장소</text><rect x="200" y="179" width="60" height="18" rx="3" fill="#8892a6"/><text x="268" y="193" font-size="12.5" fill="#555">1건</text><text x="190" y="226" font-size="13" text-anchor="end" fill="#1f2733">구조 · 사용자</text><rect x="200" y="213" width="120" height="18" rx="3" fill="#2f6fdb"/><text x="328" y="227" font-size="12.5" fill="#555">2건</text><text x="190" y="260" font-size="13" text-anchor="end" fill="#1f2733">판정 · 저장소</text><rect x="200" y="247" width="180" height="18" rx="3" fill="#8892a6"/><text x="388" y="261" font-size="12.5" fill="#555">3건</text><text x="190" y="294" font-size="13" text-anchor="end" fill="#1f2733">판정 · 사용자</text><rect x="200" y="281" width="120" height="18" rx="3" fill="#2f6fdb"/><text x="328" y="295" font-size="12.5" fill="#555">2건</text><text x="190" y="328" font-size="13" text-anchor="end" fill="#1f2733">게이트 · 저장소</text><rect x="200" y="315" width="60" height="18" rx="3" fill="#8892a6"/><text x="268" y="329" font-size="12.5" fill="#555">1건</text><text x="190" y="362" font-size="13" text-anchor="end" fill="#1f2733">게이트 · 사용자</text><rect x="200" y="349" width="60" height="18" rx="3" fill="#2f6fdb"/><text x="268" y="363" font-size="12.5" fill="#555">1건</text><text x="0" y="398" font-size="11.5" fill="#666">어휘 7건과 안내 3건은 100% 사용자 문장에서만 나왔다.</text></svg>

**종류의 분포가 숫자보다 중요하다.** 저장소 프롬프트가 찾은 5건은 전부 판정·구조·게이트
결함이고, 모두 *새 요청 형태를 일부러 만들어 넣었을 때* 나왔다. 어휘 결함은 단 한 건도
찾지 못했다 — 찾을 수 없었다.

---

## 4. 디코더 마스킹으로 읽기

이 현상은 트랜스포머 디코더의 **마스킹**과 구조가 같다. 비유가 아니라 같은 실패
양식이다 — 아래 다섯 가지가 하나씩 대응한다.

### 4.1 마스크는 오류가 아니다 · 조용한 재정규화

디코더에서 마스킹은 어텐션 로짓에 `-inf`를 더해 softmax 이후 가중치를 0으로 만든다.
중요한 건 **마스크된 위치가 예외를 던지지 않는다**는 점이다. softmax는 살아남은
항목들로 다시 정규화해 **합이 1인 정상적인 분포**를 만들고, 모델은 평소와 똑같이
확신에 찬 출력을 낸다.

<svg xmlns="http://www.w3.org/2000/svg" width="760" height="230" font-family="system-ui,Segoe UI,sans-serif"><text x="0" y="20" font-size="15" font-weight="600" fill="#1f2733">마스크는 오류를 내지 않는다 — 남은 것으로 재정규화한다</text><text x="0" y="46" font-size="13" font-weight="600" fill="#444">마스크 전 — 올바른 해석이 가장 큰 가중치</text><text x="215" y="76" font-size="12.5" text-anchor="end" fill="#1f2733">ad로 → A/D 스킴</text><rect x="225" y="62" width="198" height="18" rx="3" fill="#d0d5dd"/><text x="431" y="76" font-size="12" fill="#555">0.55</text><line x1="225" y1="71" x2="423" y2="71" stroke="#8b929d" stroke-width="1.4"/><text x="215" y="106" font-size="12.5" text-anchor="end" fill="#1f2733">방향키 → 화살표</text><rect x="225" y="92" width="108" height="18" rx="3" fill="#2f6fdb"/><text x="341" y="106" font-size="12" fill="#555">0.30</text><text x="215" y="136" font-size="12.5" text-anchor="end" fill="#1f2733">기타</text><rect x="225" y="122" width="54" height="18" rx="3" fill="#2f6fdb"/><text x="287" y="136" font-size="12" fill="#555">0.15</text><text x="0" y="152" font-size="13" font-weight="600" fill="#444">마스크 후 — 합은 여전히 1.00, 답은 확신에 참</text><text x="215" y="182" font-size="12.5" text-anchor="end" fill="#1f2733">방향키 → 화살표</text><rect x="225" y="168" width="241" height="18" rx="3" fill="#2f6fdb"/><text x="474" y="182" font-size="12" fill="#555">0.67</text><text x="215" y="212" font-size="12.5" text-anchor="end" fill="#1f2733">기타</text><rect x="225" y="198" width="118" height="18" rx="3" fill="#2f6fdb"/><text x="351" y="212" font-size="12" fill="#555">0.33</text><text x="0" y="224" font-size="11.5" fill="#666">하네스는 &quot;모르겠다&quot;고 하지 않았다. 방향키로 측정하고 verified를 찍었다.</text></svg>

하네스가 정확히 이렇게 동작했다. `ad로`가 `_AD_SCHEME`에 없으니 그 해석은 로짓
`-inf`였고, 남아 있던 `방향키`(사용자가 "방향키 **방향으로** 가속"이라고 쓴 그 단어)가
재정규화되어 1위가 됐다. 하네스는 **방향키로 이동을 측정하고 `verified`를 찍었다.**
A/D로 올바르게 구현한 게임을 떨어뜨릴 수 있는 상태였는데 경고는 한 줄도 없었다.

> 마스크된 정보는 "없는 정보"가 아니라 **"없는 것으로 취급되는 정보"**다.
> 시스템은 그 차이를 스스로 알 수 없다.

### 4.2 패딩 마스크 = 어휘 밖 표기

패딩 마스크는 시퀀스에 물리적으로 존재하지만 어텐션에서 제외되는 위치를 만든다.
사용자 문장의 토큰 중 상당수가 추출기에게는 정확히 그런 위치였다.

<svg xmlns="http://www.w3.org/2000/svg" width="820" height="132" font-family="system-ui,Segoe UI,sans-serif"><text x="0" y="20" font-size="15" font-weight="600" fill="#1f2733">추출기가 실제로 &quot;볼 수 있었던&quot; 토큰 (v1.11.13 시점)</text><text x="0" y="40" font-size="12" fill="#666">회색 = 마스크되어 존재하지 않는 것과 같은 토큰</text><rect x="0" y="58" width="36" height="30" rx="4" fill="#d0d5dd" stroke="#aab1bd" stroke-width="1"/><text x="18.0" y="78" font-size="12.5" text-anchor="middle" fill="#7b828d">새씬</text><line x1="5" y1="83" x2="31" y2="63" stroke="#98a0ac" stroke-width="1.4"/><rect x="41" y="58" width="30" height="30" rx="4" fill="#eaf0fb" stroke="#2f6fdb" stroke-width="1"/><text x="56.0" y="78" font-size="12.5" text-anchor="middle" fill="#1f2733">을</text><rect x="76" y="58" width="47" height="30" rx="4" fill="#eaf0fb" stroke="#2f6fdb" stroke-width="1"/><text x="99.5" y="78" font-size="12.5" text-anchor="middle" fill="#1f2733">열어서</text><rect x="128" y="58" width="30" height="30" rx="4" fill="#eaf0fb" stroke="#2f6fdb" stroke-width="1"/><text x="143.0" y="78" font-size="12.5" text-anchor="middle" fill="#1f2733">,</text><rect x="163" y="58" width="58" height="30" rx="4" fill="#eaf0fb" stroke="#2f6fdb" stroke-width="1"/><text x="192.0" y="78" font-size="12.5" text-anchor="middle" fill="#1f2733">2.5D</text><rect x="226" y="58" width="58" height="30" rx="4" fill="#eaf0fb" stroke="#2f6fdb" stroke-width="1"/><text x="255.0" y="78" font-size="12.5" text-anchor="middle" fill="#1f2733">플랫포머</text><rect x="289" y="58" width="36" height="30" rx="4" fill="#eaf0fb" stroke="#2f6fdb" stroke-width="1"/><text x="307.0" y="78" font-size="12.5" text-anchor="middle" fill="#1f2733">게임</text><rect x="330" y="58" width="36" height="30" rx="4" fill="#d0d5dd" stroke="#aab1bd" stroke-width="1"/><text x="348.0" y="78" font-size="12.5" text-anchor="middle" fill="#7b828d">ad</text><line x1="335" y1="83" x2="361" y2="63" stroke="#98a0ac" stroke-width="1.4"/><rect x="371" y="58" width="30" height="30" rx="4" fill="#d0d5dd" stroke="#aab1bd" stroke-width="1"/><text x="386.0" y="78" font-size="12.5" text-anchor="middle" fill="#7b828d">로</text><line x1="376" y1="83" x2="396" y2="63" stroke="#98a0ac" stroke-width="1.4"/><rect x="406" y="58" width="36" height="30" rx="4" fill="#eaf0fb" stroke="#2f6fdb" stroke-width="1"/><text x="424.0" y="78" font-size="12.5" text-anchor="middle" fill="#1f2733">좌우</text><rect x="447" y="58" width="69" height="30" rx="4" fill="#eaf0fb" stroke="#2f6fdb" stroke-width="1"/><text x="481.5" y="78" font-size="12.5" text-anchor="middle" fill="#1f2733">space</text><rect x="521" y="58" width="30" height="30" rx="4" fill="#eaf0fb" stroke="#2f6fdb" stroke-width="1"/><text x="536.0" y="78" font-size="12.5" text-anchor="middle" fill="#1f2733">로</text><rect x="556" y="58" width="36" height="30" rx="4" fill="#eaf0fb" stroke="#2f6fdb" stroke-width="1"/><text x="574.0" y="78" font-size="12.5" text-anchor="middle" fill="#1f2733">점프</text><rect x="597" y="58" width="47" height="30" rx="4" fill="#d0d5dd" stroke="#aab1bd" stroke-width="1"/><text x="620.5" y="78" font-size="12.5" text-anchor="middle" fill="#7b828d">쉬프트</text><line x1="602" y1="83" x2="639" y2="63" stroke="#98a0ac" stroke-width="1.4"/><rect x="649" y="58" width="30" height="30" rx="4" fill="#eaf0fb" stroke="#2f6fdb" stroke-width="1"/><text x="664.0" y="78" font-size="12.5" text-anchor="middle" fill="#1f2733">를</text><rect x="684" y="58" width="47" height="30" rx="4" fill="#eaf0fb" stroke="#2f6fdb" stroke-width="1"/><text x="707.5" y="78" font-size="12.5" text-anchor="middle" fill="#1f2733">누르면</text><rect x="0" y="96" width="36" height="30" rx="4" fill="#d0d5dd" stroke="#aab1bd" stroke-width="1"/><text x="18.0" y="116" font-size="12.5" text-anchor="middle" fill="#7b828d">가속</text><line x1="5" y1="121" x2="31" y2="101" stroke="#98a0ac" stroke-width="1.4"/><rect x="41" y="96" width="47" height="30" rx="4" fill="#eaf0fb" stroke="#2f6fdb" stroke-width="1"/><text x="64.5" y="116" font-size="12.5" text-anchor="middle" fill="#1f2733">카메라</text><rect x="93" y="96" width="30" height="30" rx="4" fill="#eaf0fb" stroke="#2f6fdb" stroke-width="1"/><text x="108.0" y="116" font-size="12.5" text-anchor="middle" fill="#1f2733">가</text><rect x="128" y="96" width="58" height="30" rx="4" fill="#eaf0fb" stroke="#2f6fdb" stroke-width="1"/><text x="157.0" y="116" font-size="12.5" text-anchor="middle" fill="#1f2733">플레이어</text><rect x="191" y="96" width="30" height="30" rx="4" fill="#eaf0fb" stroke="#2f6fdb" stroke-width="1"/><text x="206.0" y="116" font-size="12.5" text-anchor="middle" fill="#1f2733">를</text><rect x="226" y="96" width="36" height="30" rx="4" fill="#d0d5dd" stroke="#aab1bd" stroke-width="1"/><text x="244.0" y="116" font-size="12.5" text-anchor="middle" fill="#7b828d">추적</text><line x1="231" y1="121" x2="257" y2="101" stroke="#98a0ac" stroke-width="1.4"/><text x="0" y="126" font-size="11.5" fill="#666">마스크된 다섯 토큰이 결함 6·8·9·12·14를 만들었다.</text></svg>

`새씬`·`ad`·`로`·`쉬프트`·`가속`·`추적` — 문장에는 분명히 있는데 추출기의
어휘(`_AD_SCHEME`, `_BOOST_CONTEXT_WORDS`, `_CAMERA_FOLLOW_WORDS`,
`FRESH_SCENE_PHRASES`)에 없어 **가중치 0**이었다. 결함 6·8·9·12·14가 전부 여기서 나왔다.

### 4.3 코잘 마스크 = 화면에만 존재하는 정보

코잘 마스크는 위치 *i*가 *j > i*를 볼 수 없게 만든다. **"어렵다"가 아니라 구조적으로
불가능**하다는 것이 요점이다.

v1.11.15의 카메라 결함이 이 종류였다. 플레이어에 붙은 1인칭 시점 카메라는 변위가
플레이어와 정확히 같아 추종 검사를 **완벽하게** 통과한다. 영수증에는 델타만 남아
사후 확인도 불가능했다. 로그·영수증·단위 테스트 어디에도 그 정보에 닿는 경로가
없었다 — 그 축은 마스크되어 있었다. **화면을 본 사람만** 알 수 있었고, 실제로
사용자가 알려줬다.

수정은 마스크를 걷는 일이었다: 카메라–플레이어 **거리**를 측정값에 추가하고
(`camera_player_gap`) 영수증에 남겼다. 이제 그 축이 어텐션 범위 안에 있다.

### 4.4 마스크 불일치 = 두 모듈이 서로 다른 어휘를 들고 있었다

인코더–디코더 구조에서 두 마스크가 어긋나면 조용히 어긋난 결과가 나온다.

`task_contract`(정책 게이트)와 `verification`(측정)이 **같은 뜻의 정규식을 각자
따로** 갖고 있었다. `ad키`를 한쪽만 알아 게이트는 `spaceKey`만 강제하고 호스트는
A/D로 측정하는 상태가 됐다(결함 13). 두 번 갈라진 뒤 어휘를 `verification` 단일
출처로 합치고, **게이트가 강제하는 키 ⊇ 호스트가 측정하는 키**를 여섯 표기에 대해
검사하는 테스트를 넣었다.

### 4.5 마스크 경계 버그 = 한글에서 깨진 `\b`

마스크 구현의 off-by-one은 학습 손실에는 잘 안 드러나고 생성 품질에서만 티가 나는
고전적 버그다. 우리 쪽 대응물은 이것이다.

```python
_AD_SCHEME = re.compile(r"\ba\s*[/,·+]\s*d\b")   # 한글 앞에서 깨진다
```

파이썬 정규식에서 한글은 **단어 문자**다. 따라서 `D로`의 `d`와 `로` 사이에는
`\b` 경계가 없고, `A, D로 이동`은 통째로 매칭에 실패한다. 같은 모듈의
`_has_word()`가 주석으로 **바로 그 이유**를 적어두고 라틴 문자에만 경계를 정의하고
있었는데, `_AD_SCHEME`은 그 교훈을 받지 못한 상태였다(결함 16).

### 4.6 그리고 근본 원인 — teacher forcing으로만 평가했다

디코더를 학습 분포(teacher forcing)에서만 평가하면 자기 생성 분포(free-running)에서
무너지는 것을 못 본다. **exposure bias**다.

저장소 프롬프트가 정확히 teacher forcing이다. `_AD_SCHEME`을 `a/d`로 정한 사람이
프롬프트도 `A/D 좌우 이동`이라고 썼다. **자기 마스크 안에서 자기를 시험**하는 것이라
통과는 보장되고 정보량은 0이다. 게다가 프롬프트 파일들은 E2E를 통과시키려고 다듬어져
왔다 — 실패하면 프롬프트를 고쳤다. 살아남은 문장은 정의상 결함을 밟지 않는 문장이다
(생존자 편향).

사용자 문장은 **다른 생성기에서 나온 표본**이다. 마스크 밖에서 오기 때문에 마스크의
존재 자체를 드러낼 수 있다. 마스킹 버그는 원리적으로 in-distribution 데이터로는
찾을 수 없다.

---

## 5. 그래서 무엇을 해야 하나

### 5.1 마스크를 넓히는 것과, 마스크 밖에서 표본을 받는 것은 다른 일이다

어휘를 넓히는 것(§4.2 대응)은 **이미 아는 축**을 넓히는 일이다. 필요하지만, 다음
미지의 표기는 여전히 마스크된다. 근본 대응은 **분포 밖 표본을 계속 받는 것**이다.

| 대응 | 성격 | 비용 | 한계 |
|---|---|---|---|
| 표기 변형을 코퍼스에 추가 | 알려진 마스크 제거 | 낮음 | 다음 미지 표기는 못 잡음 |
| 정적 추출을 E2E 전에 실행 | 마스크를 **보이게** 함 | 매우 낮음 | 판정 결함은 못 잡음 |
| 사용자 문장 수집 | 분포 밖 표본 확보 | 사용자 시간 | 표본이 드묾 |
| 측정 축 추가(예: `camera_player_gap`) | 코잘 마스크 제거 | 중간 | 사례가 있어야 정당함 |

### 5.2 정적 추출은 E2E보다 100~300배 싸다

E2E 한 번은 60~200초, 정적 추출은 1초 미만이다. 새 문장을 받으면 **돌리기 전에**
무엇이 측정될지부터 본다.

```bash
uv run python -c "import sys; from verification import VerificationSpec; s=VerificationSpec.from_request(sys.argv[1]); print('검사:', s.requested_checks()); print('이동키:', s.move_right_key+'/'+s.move_left_key); print('미매핑:', s.unmapped_requirements())" "wasd로 움직이고 시프트로 대시"
```

검사 목록이 비어 있거나 이동키가 요청과 다르면 **그 자리에서 마스크를 발견한 것**이다.

### 5.3 마스크된 축을 영수증에 남긴다

v1.11.14의 `unmapped_requirements`와 v1.11.15의 `camera_player_gap`이 같은 사상이다.
못 재는 것을 **못 잰다고 기록**하면, 조용한 재정규화(§4.1)가 조용하지 않게 된다.
`verified` 옆에 "요청에 있으나 측정하지 않은 항목"이 붙는 것이 이 변화다.

### 5.4 다음 표본으로 가장 값어치 있는 것

아직 밟지 않은 축일수록 크다.

- 이동 밖의 게임 로직 — 적·충돌·점수·체력·아이템 획득
- 기존 씬을 **수정**하는 요청 (지금까지는 전부 새 씬 생성이었다)
- 한국어/영어 혼용, 축약, 오타가 있는 자연스러운 문장

---

## 6. 한계

- **표본이 작다.** 사용자 문장 3개, 결함 20건. 15 대 5라는 비율보다 §3의 **종류
  분포**(어휘·안내 결함이 100% 사용자 쪽)가 더 단단한 근거다.
- 결함의 출처 분류는 사람이 했다. 어떤 결함은 두 코퍼스 모두에서 나올 수 있었다.
- 저장소 프롬프트가 무용한 것은 아니다. **새 요청 형태를 일부러 만들었을 때는** 판정·
  구조 결함 5건을 찾았다. 이미 다루는 형태 안에서 무력할 뿐이다.
- 마스킹 비유는 실패 **양식**의 대응이지, 하네스가 신경망이라는 뜻이 아니다. 하네스의
  마스크는 학습된 것이 아니라 사람이 정규식으로 손으로 쓴 것이고, 그래서 **고칠 수
  있다** — 이 보고서의 결론이 비관적이지 않은 이유다.
